In [ ]:
import pandas as pd
import numpy as np
import psycopg2 as pg

# Database connection details (host, port, credentials) are institution-specific and are not included in this repository.
# Users should configure their own connection to an OMOP CDM-formatted database.

Conn = pg.connect(host = "<your_host>",
                 port = "<your_port>",
                 dbname = "<your_dbname>",
                 user = "<your_user>",
                 password = "<your_password>"
)
cursor = Conn.cursor()

# Schema names: replace with the schemas of your OMOP CDM database
cdm_schema = "<your_cdm_schema>"
vocab_schema = "<your_vocab_schema>"

def render(sql):
    return sql.replace("@cdm_schema", cdm_schema).replace("@vocab_schema", vocab_schema)

# Phase 1. Generation of the local-OHDSI SNOMED CT concept pairs

In [ ]:
# Base_diagnosis data (diagnosis records whose source concept belongs to the ICD10 vocabulary)
query_base ="""
select
    co.condition_occurrence_id,
    co.condition_source_value as local_code
from @cdm_schema.condition_occurrence co
join @vocab_schema.concept c
on co.condition_source_concept_id = c.concept_id
where c.vocabulary_id = 'ICD10'
"""

df_base = pd.read_sql_query(render(query_base), Conn)

In [ ]:
# Locally-mapped SNOMED CT Codes
# source_code: local diagnosis code / source_concept_id: ICD-10 concept / target_concept_id: locally mapped SNOMED CT concept
query_locally ="""
select
    m.source_code as local_code,
    m.source_concept_id as icd_concept_id,
    m.target_concept_id as local_snomed_concept_id
from @vocab_schema.source_to_concept_map m
join @vocab_schema.concept c
on m.target_concept_id = c.concept_id
where m.invalid_reason is null
and m.target_vocabulary_id = 'SNOMED'
and c.standard_concept = 'S'
"""

df_locally_raw = pd.read_sql_query(render(query_locally), Conn)
df_with_local = df_base.merge(df_locally_raw, on="local_code", how="inner")

In [ ]:
# OHDSI-mapped SNOMED CT Codes
query_ohdsi = """
select
    r.concept_id_1 as icd_concept_id,
    r.concept_id_2 as ohdsi_snomed_concept_id
from @vocab_schema.concept_relationship r
join @vocab_schema.concept c
  on r.concept_id_2 = c.concept_id
where r.relationship_id = 'Maps to'
  and c.vocabulary_id = 'SNOMED'
  and c.standard_concept = 'S'
"""

df_ohdsi_raw = pd.read_sql_query(render(query_ohdsi), Conn)
df_with_both_maps = df_with_local.merge(df_ohdsi_raw, on="icd_concept_id", how="inner")

df_pairs = df_with_both_maps[['icd_concept_id', 'local_snomed_concept_id', 'ohdsi_snomed_concept_id']].drop_duplicates()

In [ ]:
# SNOMED CT concept ancestor relationships
query_ancestor = """
select ca.ancestor_concept_id, ca.descendant_concept_id, ca.min_levels_of_separation
from @vocab_schema.concept_ancestor ca
join @vocab_schema.concept c
on ca.ancestor_concept_id = c.concept_id
where c.vocabulary_id = 'SNOMED'
"""

df_ancestor = pd.read_sql_query(render(query_ancestor), Conn)

# SNOMED CT concept names (used to identify generic shared ancestors in Phase 3)
query_concept = """
select concept_id, concept_name
from @vocab_schema.concept
where vocabulary_id = 'SNOMED'
"""

df_concept = pd.read_sql_query(render(query_concept), Conn)

# Phase 2. Classification of hierarchical relationships

In [ ]:
# Type 1: Exact match
df_classified = df_pairs.copy()

same_keys = set(
    df_classified[
        df_classified['local_snomed_concept_id'] == df_classified['ohdsi_snomed_concept_id']
    ][['icd_concept_id', 'local_snomed_concept_id']].itertuples(index=False, name=None)
)

# Type 2: IS-A Relation (One concept is an ancestor of the other)
df_anc1 = df_pairs.merge(df_ancestor, left_on = ['ohdsi_snomed_concept_id', 'local_snomed_concept_id'],
                       right_on = ['ancestor_concept_id', 'descendant_concept_id'], how = 'left', indicator = True).rename(columns={'_merge':'match1'})
df_anc2 = df_pairs.merge(df_ancestor, left_on = ['local_snomed_concept_id', 'ohdsi_snomed_concept_id'],
                       right_on = ['ancestor_concept_id', 'descendant_concept_id'], how = 'left', indicator = True).rename(columns={'_merge':'match2'})


def classify(row):
    if (row['icd_concept_id'], row['local_snomed_concept_id']) in same_keys:
        return 'exact_match'
    elif ((df_anc1['match1'] == 'both') &
          (df_anc1['icd_concept_id'] == row['icd_concept_id']) &
          (df_anc1['ohdsi_snomed_concept_id'] == row['ohdsi_snomed_concept_id']) &
          (df_anc1['local_snomed_concept_id'] == row['local_snomed_concept_id'])).any():
        return 'ohdsi_in_local'
    elif ((df_anc2['match2'] == 'both') &
          (df_anc2['icd_concept_id'] == row['icd_concept_id']) &
          (df_anc2['ohdsi_snomed_concept_id'] == row['ohdsi_snomed_concept_id']) &
          (df_anc2['local_snomed_concept_id'] == row['local_snomed_concept_id'])).any():
        return 'local_in_ohdsi'
    else:
        return 'no_direct_relation'

df_classified['relation_type'] = df_classified.apply(classify, axis=1)

In [ ]:
# no_direct_relation cases: evaluated for a shared common ancestor
df_no_relation = df_classified[df_classified['relation_type'] == 'no_direct_relation'].copy()

ohdsi_ids = df_no_relation['ohdsi_snomed_concept_id'].unique().tolist()
local_ids = df_no_relation['local_snomed_concept_id'].unique().tolist()

df_ohdsi_ancestors = df_ancestor[df_ancestor['descendant_concept_id'].isin(ohdsi_ids)].copy()
df_local_ancestors = df_ancestor[df_ancestor['descendant_concept_id'].isin(local_ids)].copy()

df_shared_anc = df_ohdsi_ancestors.merge(
    df_local_ancestors,
    on='ancestor_concept_id',
    how='inner',
    suffixes=('_ohdsi','_local')
)

df_with_anc = df_no_relation.merge(
    df_shared_anc[['descendant_concept_id_ohdsi','descendant_concept_id_local',
                   'ancestor_concept_id', 'min_levels_of_separation_ohdsi', 'min_levels_of_separation_local']],
    left_on = ['ohdsi_snomed_concept_id', 'local_snomed_concept_id'],
    right_on = ['descendant_concept_id_ohdsi', 'descendant_concept_id_local'],
    how='left'
)

In [ ]:
# Nearest shared ancestor per pair, then final Type 1-4 classification
df_with_anc['total_level'] = df_with_anc['min_levels_of_separation_ohdsi']+df_with_anc['min_levels_of_separation_local']
df_with_anc['rn'] = df_with_anc.groupby(
    ['icd_concept_id', 'ohdsi_snomed_concept_id', 'local_snomed_concept_id']
)['total_level'].rank(method='first')

df_nearest_anc= df_with_anc[(df_with_anc['rn'] == 1) | (df_with_anc['rn'].isna())].copy()

df_classified = df_classified.merge(
    df_nearest_anc[['icd_concept_id', 'ohdsi_snomed_concept_id', 'local_snomed_concept_id',
                    'ancestor_concept_id', 'min_levels_of_separation_ohdsi', 'min_levels_of_separation_local',
                    'total_level']],
    on=['icd_concept_id', 'ohdsi_snomed_concept_id', 'local_snomed_concept_id'],
    how='left'
)

# Name of the nearest shared ancestor
df_classified = df_classified.merge(
    df_concept.rename(columns={'concept_id': 'ancestor_concept_id'}),
    on='ancestor_concept_id',
    how='left'
)

def assign_type(row):
    if row['relation_type'] =='exact_match':
        return 1
    elif row['relation_type'] in ('ohdsi_in_local', 'local_in_ohdsi'):
        return 2
    elif pd.notna(row['ancestor_concept_id']):
        return 3
    return 4

df_classified['phase2_type'] = df_classified.apply(assign_type, axis=1)

# Phase 3. Automated quality categorization

In [ ]:
# Type 1 is concordant and excluded from further evaluation; Type 2-4 pairs proceed to QC assignment
df_qc = df_classified[df_classified['phase2_type'] != 1].copy()

In [ ]:
# Type 2: QC assignment by hierarchical distance (IS-A relation, no shared-ancestor distance applicable)
# distant(l-o): direct hierarchical distance between the locally-mapped and OHDSI-mapped concepts,
# taken from whichever of df_anc1/df_anc2 (Phase 2) matched as 'both'
df_type2 = df_qc[df_qc['phase2_type'] == 2].copy()

df_type2 = df_type2.merge(
    df_anc1[['icd_concept_id', 'ohdsi_snomed_concept_id', 'local_snomed_concept_id', 'min_levels_of_separation']],
    on=['icd_concept_id', 'ohdsi_snomed_concept_id', 'local_snomed_concept_id'],
    how='left'
).merge(
    df_anc2[['icd_concept_id', 'ohdsi_snomed_concept_id', 'local_snomed_concept_id', 'min_levels_of_separation']],
    on=['icd_concept_id', 'ohdsi_snomed_concept_id', 'local_snomed_concept_id'],
    how='left', suffixes=('_anc1', '_anc2')
)

df_type2['distant(l-o)'] = df_type2['min_levels_of_separation_anc1'].fillna(df_type2['min_levels_of_separation_anc2'])

df_type2['qc'] = np.where(df_type2['distant(l-o)'] <= 3, 'QC1', 'QC2')

In [ ]:
# Type 3: QC assignment combining structural distance and attribute-based similarity
df_type3 = df_qc[df_qc['phase2_type'] == 3].copy()

df_type3 = df_type3.rename(columns={
    'min_levels_of_separation_ohdsi': 'level_from_ohdsi',
    'min_levels_of_separation_local': 'level_from_local',
    'total_level': 'total_distance'
})

# Excessive-distance thresholds (0.99 quantile), estimated from type 3 pairs
q99_o = df_type3['level_from_ohdsi'].quantile(0.99)
q99_l = df_type3['level_from_local'].quantile(0.99)

generic_ancestor_list = ['Disease', 'Clinical finding', 'Disorder of body system']

quantile_results = {}
for anc in generic_ancestor_list:
    subset = df_type3[df_type3['concept_name'] == anc]
    quantile_results[anc] = subset['total_distance'].quantile(0.99)

In [ ]:
# Attribute values for Jaccard similarity (has_finding_site, has_associated_morphology)
query_relationship = """
select
    concept_id_1,
    concept_id_2,
    relationship_id
from @vocab_schema.concept_relationship
where relationship_id IN ('Has finding site', 'Has asso morph')
"""

df_rel = pd.read_sql_query(render(query_relationship), Conn)
attribute_dict = df_rel.groupby('concept_id_1')['concept_id_2'].apply(set).to_dict()

In [ ]:
# Jaccard similarity: a score of zero is assigned when either attribute set is missing for one or both concepts
def jaccard_similarity(set1, set2):
    if not set1 or not set2:
        return 0.0
    return len(set1 & set2) / len(set1 | set2)

jaccard_scores = []
for idx, row in df_type3.iterrows():
    local_set = attribute_dict.get(row['local_snomed_concept_id'], set())
    ohdsi_set = attribute_dict.get(row['ohdsi_snomed_concept_id'], set())
    jaccard_scores.append(jaccard_similarity(local_set, ohdsi_set))

df_type3['jaccard_similarity'] = jaccard_scores

In [ ]:
# QC 3 conditions
cond_qc3_no_ancestor = df_type3['ancestor_concept_id'].isna()
cond_qc3_distance = (df_type3['level_from_ohdsi'] >= q99_o) | (df_type3['level_from_local'] >= q99_l)
cond_qc3_generic = df_type3.apply(
    lambda x: x['concept_name'] in generic_ancestor_list and
              x['total_distance'] >= quantile_results[x['concept_name']],
    axis=1
)

# QC 1 condition
cond_qc1 = (
    ~df_type3['concept_name'].isin(generic_ancestor_list) &
    (df_type3['level_from_ohdsi'] <= 3) & (df_type3['level_from_local'] <= 3)
)

df_type3['qc'] = np.select(
    [cond_qc3_no_ancestor, cond_qc3_distance, cond_qc3_generic, cond_qc1],
    ['QC3', 'QC3', 'QC3', 'QC1'],
    default='QC2'
)

# Jaccard similarity of 0 downgrades a QC1 candidate to QC2
df_type3.loc[
    (df_type3['qc'] != 'QC3') & (df_type3['jaccard_similarity'] == 0),
    'qc'
] = 'QC2'

In [ ]:
# Type 4: no shared ancestor below the SNOMED CT root concept — automatically assigned to QC 3
df_type4 = df_qc[df_qc['phase2_type'] == 4].copy()
df_type4['qc'] = 'QC3'

# Combine Type 2, 3, 4 into the final QC-classified dataset; columns not applicable to a given type remain NaN
df_qc = pd.concat([df_type2, df_type3, df_type4], ignore_index=True)